In [ ]:
# imports, wd and seed

from pathlib import Path
import os
import re
import numpy as np
import pandas as pd
import datasets
from datasets import Dataset
import random 
import sys
import matplotlib.pyplot as plt

import torch
torch.backends.mps.is_available(), torch.backends.mps.is_built()

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score, classification_report

import transformers
transformers.__version__
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed
)

from scipy.stats import mannwhitneyu
import statsmodels.api as sm


# setting seed for reproducibility (to the degree that is possible with MPS)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
set_seed(SEED)
torch.manual_seed(SEED)
if torch.backends.mps.is_available():
    torch.mps.manual_seed(SEED)


# project root
PROJECT_ROOT = Path.cwd().parent
print("Project root:", PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

In [ ]:
# checking if mps is available on local comp
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
device

In [ ]:
# loading samples and creating label lists

samples_path = Path("data/generated/raw/samples.jsonl")
df = pd.read_json(samples_path, lines=True)

# labels column is like: "temperature+pressure"
df["label_list"] = df["labels"].str.split("+")
df["k_mods"] = df["label_list"].apply(len)

df.head()

In [ ]:
# train/val/test splitting by job_id

job_ids = df["job_id"].unique()

rng = np.random.default_rng(SEED)
rng.shuffle(job_ids)

n = len(job_ids)
train_ids = set(job_ids[: int(0.8 * n)])
val_ids   = set(job_ids[int(0.8 * n): int(0.9 * n)])
test_ids  = set(job_ids[int(0.9 * n):])

def assign_split(j):
    if j in train_ids: return "train"
    if j in val_ids:   return "val"
    return "test"

df["split"] = df["job_id"].map(assign_split)
split_path = Path(f"data/processed/splits_jobid_seed{SEED}.csv")
split_path.parent.mkdir(parents=True, exist_ok=True)
df[["job_id","split"]].drop_duplicates().to_csv(split_path, index=False)


df["split"].value_counts(), df["split"].value_counts(normalize=True)


In [ ]:
# sanity checkk - works!

df.groupby("split")["label_list"].apply(lambda s: pd.Series([x for xs in s for x in xs]).value_counts())

In [7]:
# binarising labels

mlb = MultiLabelBinarizer()
mlb.fit(df["label_list"])

mlb.classes_

# save label order
np.save("data/processed/label_classes.npy", mlb.classes_)

In [9]:
def add_label_matrix(df_in: pd.DataFrame) -> pd.DataFrame:
    out = df_in.copy()
    Y = mlb.transform(out["label_list"])
    out["label_vec"] = list(Y.astype(np.float32))
    return out

train_df = add_label_matrix(df[df["split"] == "train"])
val_df   = add_label_matrix(df[df["split"] == "val"])
test_df  = add_label_matrix(df[df["split"] == "test"])


In [ ]:
# huggingface dataset andd tokenizer 

model_name = "distilroberta-base"

# running only to log exact model SHA for revision - already have it, DO NOT RERUN
#from huggingface_hub import HfApi
#api = HfApi()
#info = api.model_info(model_name)
#print("Model:", info.modelId)
#print("SHA:", info.sha)

REVISION = "fb53ab8802853c8e4fbdbcd0529f21fc6f459b2b"

tokenizer = AutoTokenizer.from_pretrained(model_name, revision=REVISION)

num_labels = len(mlb.classes_)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    revision=REVISION,
    num_labels=num_labels,
    problem_type="multi_label_classification"
)

max_length = 64

def to_hf_dataset(df_in: pd.DataFrame) -> Dataset:
    df_in = df_in.reset_index(drop=True)
    return Dataset.from_dict({
        "text": df_in["text"].tolist(),
        "labels": df_in["label_vec"].tolist(),
        "anchor_regime": df_in["anchor_regime"].tolist(),
        "literal": df_in["literal"].tolist(),
        "specificity": df_in["specificity"].tolist(),
        "consistent": df_in["consistent"].tolist(),
        "k_mods": df_in["k_mods"].tolist(),
    })


train_ds = to_hf_dataset(train_df)
val_ds   = to_hf_dataset(val_df)
test_ds  = to_hf_dataset(test_df)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=max_length)

train_ds = train_ds.map(tokenize, batched=True)
val_ds   = val_ds.map(tokenize, batched=True)
test_ds  = test_ds.map(tokenize, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=None)


In [ ]:
print("Model name_or_path:", model.config._name_or_path)
print("Num labels:", model.config.num_labels)
print("Hidden layers:", getattr(model.config, "num_hidden_layers", None))


In [13]:
# metrics (micro and macro F1)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))  # sigmoid
    preds = (probs >= 0.5).astype(int)

    micro = f1_score(labels, preds, average="micro", zero_division=0)
    macro = f1_score(labels, preds, average="macro", zero_division=0)
    return {"micro_f1": micro, "macro_f1": macro}


In [ ]:
# TrainingArguments tuned for MPS

training_args = TrainingArguments(
    output_dir="models/distilroberta_multilabel",
    learning_rate=2e-5,
    per_device_train_batch_size=16,   
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="micro_f1",
    greater_is_better=True,
    logging_steps=50,
    seed=SEED,
    data_seed=SEED,
    report_to="none"
)


In [ ]:
# for logging environment info below 
ckpt = Path("models/distilroberta_multilabel") / "checkpoint-1384"
bin_path = ckpt / "training_args.bin"

if bin_path.exists():
    args = torch.load(bin_path, weights_only=False)
    out_path = Path("models/distilroberta_multilabel/first_run_training_args.json")
    out_path.write_text(json.dumps(args.to_dict(), indent=2))
    print("Loaded TrainingArguments from:", ckpt)
    print("Wrote:", out_path)
else:
    print("No checkpoint training_args.bin found yet (fresh run). Skipping checkpoint-args export.")



In [ ]:
# logging environment info 

runinfo = Path("models/distilroberta_multilabel/run_info.txt")
runinfo.parent.mkdir(parents=True, exist_ok=True)

runinfo.write_text(
    f"python={sys.version}\n"
    f"torch={torch.__version__}\n"
    f"transformers={transformers.__version__}\n"
    f"datasets={datasets.__version__}\n"
    f"device={device}\n"
    f"seed={SEED}\n"
    f"model_name={model_name}\n"
    f"revision={REVISION}\n"
    f"max_length={max_length}\n"
    f"num_labels={num_labels}\n"
    f"training_args={training_args.to_dict()}\n"
)

print("Run info written to:", runinfo)

In [ ]:
# traininggg

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
# eval on test plus per-label report

pred = trainer.predict(test_ds)
logits = pred.predictions
probs = 1 / (1 + np.exp(-logits))
Y_pred = (probs >= 0.5).astype(int)

Y_true = np.array(test_df["label_vec"].tolist()).astype(int)

print("TEST micro F1:", f1_score(Y_true, Y_pred, average="micro", zero_division=0))
print("TEST macro F1:", f1_score(Y_true, Y_pred, average="macro", zero_division=0))
print()

print("Per-label report (TEST):")
print(classification_report(Y_true, Y_pred, target_names=mlb.classes_, zero_division=0))


In [ ]:
leak_terms = ["nociception", "temperature", "pressure", "vibration", "dataset", "label"]
for t in leak_terms:
    n = df["text"].str.lower().str.contains(t).sum()
    print(t, n)

In [ ]:
# testing whether the appearance of the labels drives  F1 scores

def mask_label_words(text):
    t = text
    t = re.sub(r"\bpressure\b", "___", t, flags=re.IGNORECASE)
    t = re.sub(r"\bvibration(s)?\b", "___", t, flags=re.IGNORECASE)
    t = re.sub(r"\btemperature\b", "___", t, flags=re.IGNORECASE)
    t = re.sub(r"\bnociception\b", "___", t, flags=re.IGNORECASE)
    return t

# clean masked texts + df
masked_texts = [mask_label_words(t) for t in test_df["text"].tolist()]

masked_test_ds = Dataset.from_dict({
    "text": masked_texts,
    "labels": [list(map(float, v)) for v in test_df["label_vec"].tolist()],
})

# tokenising
def tokenize_fn(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=64,
    )

masked_test_ds = masked_test_ds.map(tokenize_fn, batched=True)

# removing raw text so tensors are the only left
masked_test_ds = masked_test_ds.remove_columns(["text"])

# which columns become torch tensors:
masked_test_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# prediction
pred_masked = trainer.predict(masked_test_ds)
logits_masked = pred_masked.predictions
probs_masked = 1 / (1 + np.exp(-logits_masked))
Y_pred_masked = (probs_masked >= 0.5).astype(int)

# original preds
Y_true = np.array(test_df["label_vec"].tolist()).astype(int)

print("Original TEST micro F1:", f1_score(Y_true, Y_pred, average="micro"))
print("Masked   TEST micro F1:", f1_score(Y_true, Y_pred_masked, average="micro"))




In [20]:
# performance breakdown by experimental axes 

test_df["pred_list"] = list(mlb.inverse_transform(Y_pred))
test_df["true_list"] = list(mlb.inverse_transform(Y_true))

def f1_for_subset(mask):
    sub = test_df[mask]
    if len(sub) == 0:
        return np.nan
    Y_t = mlb.transform(sub["true_list"])
    Y_p = mlb.transform(sub["pred_list"])
    return f1_score(Y_t, Y_p, average="micro")



### FULL EVAL BELOW

In [ ]:

#assert hasattr(mlb, "classes_"), "mlb must be fitted earlier; do not refit it here."


# building test_df
test_df = df.loc[df["split"].eq("test")].copy().reset_index(drop=True)

if "label_list" not in test_df.columns:
    test_df["label_list"] = test_df["labels"].fillna("").astype(str).str.split("+")

test_df["label_list"] = test_df["label_list"].apply(lambda xs: [x for x in xs if isinstance(x, str) and x.strip() != ""])


# k-modality count based on label_list 
test_df["k_mods"] = test_df["label_list"].apply(len)

if "label_vec" not in test_df.columns:
    test_df["label_vec"] = list(mlb.transform(test_df["label_list"]).astype(np.float32))

Y_test_true = np.array(test_df["label_vec"].tolist()).astype(int)


# one consistent pass on each set
tmp_test_ds = Dataset.from_dict({
    "text": test_df["text"].tolist(),
    "labels": [list(map(float, v)) for v in test_df["label_vec"].tolist()],
})

def _tokenize_for_eval(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=max_length,
    )
tmp_test_ds = tmp_test_ds.map(_tokenize_for_eval, batched=True)
tmp_test_ds = tmp_test_ds.remove_columns(["text"])
tmp_test_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

pred = trainer.predict(tmp_test_ds)
logits = pred.predictions
probs = 1 / (1 + np.exp(-logits))
Y_test_pred = (probs >= 0.5).astype(int)


# converting to label lists
test_df["true_list"] = list(mlb.inverse_transform(Y_test_true))
test_df["pred_list"] = list(mlb.inverse_transform(Y_test_pred))

print("Transformer TEST micro F1:", f1_score(Y_test_true, Y_test_pred, average="micro", zero_division=0))
print("Transformer TEST macro F1:", f1_score(Y_test_true, Y_test_pred, average="macro", zero_division=0))


In [ ]:
# helpers (micro-f1 on subset)
def micro_f1_subset(mask):
    mask = np.array(mask)
    if mask.sum() == 0:
        return np.nan
    sub_true = mlb.transform(test_df.loc[mask, "true_list"])
    sub_pred = mlb.transform(test_df.loc[mask, "pred_list"])
    return f1_score(sub_true, sub_pred, average="micro", zero_division=0)

def summarise_by(col, values=None):
    out = []
    if values is None:
        values = sorted(test_df[col].dropna().unique().tolist())
    for v in values:
        m = test_df[col].eq(v)
        out.append({"axis": col, "value": v, "n": int(m.sum()), "micro_f1": micro_f1_subset(m)})
    return pd.DataFrame(out)

# songle-axis breakdowns
tables = []

if "anchor_regime" in test_df.columns:
    tables.append(summarise_by("anchor_regime", ["strict", "paraphrase", "drift"]))
if "literal" in test_df.columns:
    tables.append(summarise_by("literal", [0, 1]))
if "specificity" in test_df.columns:
    tables.append(summarise_by("specificity", [0, 1]))
if "consistent" in test_df.columns:
    tables.append(summarise_by("consistent", [0, 1]))

tables.append(summarise_by("k_mods", [1, 2, 3]))
axis_table_transformer = pd.concat(tables, ignore_index=True)
axis_table_transformer


In [ ]:
def interaction_row(name, mask):
    mask = np.array(mask)
    return {"group": name, "n": int(mask.sum()), "micro_f1": micro_f1_subset(mask)}

rows = []

if "anchor_regime" in test_df.columns and "literal" in test_df.columns:
    for r in ["strict", "paraphrase", "drift"]:
        for lit in [0, 1]:
            rows.append(interaction_row(
                f"{r} & literal={lit}",
                (test_df["anchor_regime"].eq(r) & test_df["literal"].eq(lit))
            ))

if "anchor_regime" in test_df.columns:
    for r in ["strict", "paraphrase", "drift"]:
        for k in [1, 2, 3]:
            rows.append(interaction_row(
                f"{r} & k_mods={k}",
                (test_df["anchor_regime"].eq(r) & test_df["k_mods"].eq(k))
            ))

if "literal" in test_df.columns:
    for lit in [0, 1]:
        for k in [1, 2, 3]:
            rows.append(interaction_row(
                f"literal={lit} & k_mods={k}",
                (test_df["literal"].eq(lit) & test_df["k_mods"].eq(k))
            ))


# 3-way mode
if "anchor_regime" in test_df.columns and "literal" in test_df.columns:
    rows.append(interaction_row(
        "drift & metaphor & k_mods=3",
        (test_df["anchor_regime"].eq("drift") & test_df["literal"].eq(0) & test_df["k_mods"].eq(3))
    ))
    rows.append(interaction_row(
        "paraphrase & metaphor & k_mods=3",
        (test_df["anchor_regime"].eq("paraphrase") & test_df["literal"].eq(0) & test_df["k_mods"].eq(3))
    ))

if "anchor_regime" in test_df.columns and "consistent" in test_df.columns:
    rows.append(interaction_row(
        "strict & inconsistent",
        (test_df["anchor_regime"].eq("strict") & test_df["consistent"].eq(0))
    ))

inter_table = (
    pd.DataFrame(rows)
      .sort_values(["micro_f1", "n"], ascending=[True, False])
      .reset_index(drop=True)
)

inter_table.head(25)

## Second part of the evaluation: Cross-modal salience and its' influence on classification

In [ ]:
# test rows from df
test_mask = df["split"].eq("test").values
test_df = df.loc[test_mask].copy().reset_index(drop=True)

# matrix for labels 
test_df["true_list"] = test_df["labels"].astype(str).str.split("+").tolist()
Y_test_true = mlb.transform(test_df["true_list"])

# transformer predictions
if "Y_test_pred" not in globals():
    tmp_test_ds = Dataset.from_dict({"text": test_df["text"].tolist()})

    def tokenize_fn(batch):
        return tokenizer(
            batch["text"],
            truncation=True,
            padding="max_length",
            max_length=max_length,
        )

    tmp_test_ds = tmp_test_ds.map(tokenize_fn, batched=True)
    tmp_test_ds = tmp_test_ds.remove_columns(["text"])
    tmp_test_ds.set_format(type="torch", columns=["input_ids", "attention_mask"])

    pred = trainer.predict(tmp_test_ds)
    logits = pred.predictions
    probs = 1 / (1 + np.exp(-logits))
    Y_test_pred = (probs >= 0.5).astype(int)

# converting predictions to label lists
test_df["pred_list"] = list(mlb.inverse_transform(Y_test_pred))
test_df["true_list"] = list(mlb.inverse_transform(Y_test_true))

strict_test = test_df[test_df["anchor_regime"].eq("strict")].copy()

strict_test["exact_match"] = strict_test.apply(
    lambda r: set(r["true_list"]) == set(r["pred_list"]), axis=1
)

strict_test[["anchors_planned","labels","exact_match"]].head()


In [26]:

lex_path = PROJECT_ROOT / "data" / "raw" / "Sensorimotor_norms_24Jan2026.csv"
lex = pd.read_csv(lex_path)

lex["Word_norm"] = lex["Word"].astype(str).str.strip().str.upper()

cols = [
    "Word_norm",
    "Haptic.mean", "Visual.mean", "Auditory.mean", "Olfactory.mean", "Gustatory.mean", "Interoceptive.mean"
]
lex_small = lex[cols].copy()
lex_lookup = lex_small.set_index("Word_norm").to_dict(orient="index")


In [27]:
def parse_anchors_planned(s):
    if pd.isna(s) or str(s).strip() == "":
        return []
    return [a.strip().upper() for a in str(s).split(";") if a.strip()]

other_modal_cols = ["Visual.mean","Auditory.mean","Olfactory.mean","Gustatory.mean","Interoceptive.mean"]

def anchor_to_scores(anchor):
    a = anchor.strip().upper()
    if a in lex_lookup:
        d = lex_lookup[a]
        h = d["Haptic.mean"]
        other_max = max(d[c] for c in other_modal_cols)
        return {"anchor": a, "found": True, "haptic": h, "other_max": other_max, "dominance": h - other_max}

    # fallback for multiword anchors: split and try parts
    parts = a.replace("-", " ").split()
    part_rows = [lex_lookup[p] for p in parts if p in lex_lookup]

    if len(part_rows) == 0:
        return {"anchor": a, "found": False, "haptic": np.nan, "other_max": np.nan, "dominance": np.nan}

    # picking the part with highest haptic mean
    best = max(part_rows, key=lambda r: r["Haptic.mean"])
    h = best["Haptic.mean"]
    other_max = max(best[c] for c in other_modal_cols)
    return {"anchor": a, "found": True, "haptic": h, "other_max": other_max, "dominance": h - other_max}

strict_test["anchors_list"] = strict_test["anchors_planned"].apply(parse_anchors_planned)


In [ ]:

def row_dominance_stats(anchor_list):
    rows = [anchor_to_scores(a) for a in anchor_list]
    doms = [r["dominance"] for r in rows if np.isfinite(r["dominance"])]
    found = sum(1 for r in rows if r["found"])
    total = len(anchor_list)

    if len(doms) == 0:
        return pd.Series({
            "n_anchors": total,
            "n_found": found,
            "dominance_mean": np.nan,
            "dominance_min": np.nan
        })

    return pd.Series({
        "n_anchors": total,
        "n_found": found,
        "dominance_mean": float(np.mean(doms)),
        "dominance_min": float(np.min(doms))
    })

strict_test[["n_anchors","n_found","dominance_mean","dominance_min"]] = strict_test["anchors_list"].apply(row_dominance_stats)
strict_test[["n_anchors","n_found","dominance_mean","dominance_min","exact_match"]].head()

In [ ]:
coverage = strict_test["n_found"].sum() / strict_test["n_anchors"].sum()
coverage

In [ ]:
strict_ok  = strict_test.loc[strict_test["exact_match"] == True,  "dominance_mean"].dropna()
strict_bad = strict_test.loc[strict_test["exact_match"] == False, "dominance_mean"].dropna()

summary = pd.DataFrame({
    "Group": ["Correctly classified", "Incorrectly classified"],
    "Mean dominance": [strict_ok.mean(), strict_bad.mean()],
    "Median dominance": [strict_ok.median(), strict_bad.median()],
    "N": [len(strict_ok), len(strict_bad)]
})
summary

In [ ]:
# Mann-whitney U-test

u_stat, p_val = mannwhitneyu(strict_ok, strict_bad, alternative="less")

print("Mann–Whitney U test (lower dominance → correct classification)")
print(f"U statistic: {u_stat:.1f}")
print(f"p-value:     {p_val:.4f}")
print(f"N correct:   {len(strict_ok)}")
print(f"N incorrect: {len(strict_bad)}")

effect_size = u_stat / (len(strict_ok) * len(strict_bad))
print(f"Rank-biserial effect size (U / (n1·n2)): {effect_size:.3f}")

In [ ]:

tmp = strict_test.dropna(subset=["dominance_mean"]).copy()
X = sm.add_constant(tmp["dominance_mean"])
y = tmp["exact_match"].astype(int)

m = sm.Logit(y, X).fit(disp=False)
m.summary()


In [ ]:
# saving axis table for plotting later 
Path("data/processed").mkdir(parents=True, exist_ok=True)

axis_table_transformer.to_csv("data/processed/axis_table_transformer.csv", index=False)
strict_test.to_csv("data/processed/strict_test_transformer.csv", index=False)
print("Saved transformer tables.")

## PLOTTING

In [47]:

axis_table = pd.read_csv("data/processed/axis_table_baseline.csv")
axis_table_transformer = pd.read_csv("data/processed/axis_table_transformer.csv")

strict_test_base = pd.read_csv("data/processed/strict_test_baseline.csv")
strict_test_tr   = pd.read_csv("data/processed/strict_test_transformer.csv")


In [ ]:
# Output dir for figures

FIG_DIR = Path("figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

axis_table.head(), axis_table_transformer.head()

In [ ]:
# Plot: micro-F1 by generation axis (TF-IDF baseline vs DistilRoBERTa)

def plot_axis_microf1_comparison_all(base, tr, out_path=None):
    base = base.copy()
    tr = tr.copy()

    for df_ in (base, tr):
        df_["axis"] = df_["axis"].astype(str)
        df_["value"] = df_["value"].astype(str)
        df_["n"] = df_["n"].astype(int)
        df_["micro_f1"] = df_["micro_f1"].astype(float)

    base["model"] = "TF-IDF + LR"
    tr["model"] = "DistilRoBERTa"
    plot_df = pd.concat([base, tr], ignore_index=True)

    axes = ["anchor_regime", "consistent", "k_mods", "literal", "specificity"]
    order_map = {
        "anchor_regime": ["strict", "paraphrase", "drift"],
        "literal": ["0", "1"],
        "specificity": ["0", "1"],
        "consistent": ["0", "1"],
        "k_mods": ["1", "2", "3"],
    }

    axes_present = [ax for ax in axes if base["axis"].eq(ax).any() and tr["axis"].eq(ax).any()]
    if len(axes_present) == 0:
        raise ValueError("No overlapping axes found between base and transformer tables.")

    ncols = 3
    nrows = int(np.ceil(len(axes_present) / ncols))
    fig, axs = plt.subplots(nrows, ncols, figsize=(16, 8.5), constrained_layout=False)
    axs = np.array(axs).reshape(-1)

    for i, ax_name in enumerate(axes_present):
        ax = axs[i]
        sub = plot_df.loc[plot_df["axis"].eq(ax_name)].copy()

        if ax_name in order_map:
            sub["value"] = pd.Categorical(sub["value"], categories=order_map[ax_name], ordered=True)
            sub = sub.sort_values(["value", "model"])

        values = sub["value"].astype(str).unique().tolist()

        pivot = (
            sub.pivot_table(index="value", columns="model", values="micro_f1", aggfunc="first")
              .reindex(values)
        )
        n_base = (
            sub.loc[sub["model"].eq("TF-IDF + LR"), ["value", "n"]]
               .drop_duplicates()
               .set_index("value")
               .reindex(values)["n"]
        )

        x = np.arange(len(values))
        width = 0.34

        ax.bar(x - width / 2, pivot["TF-IDF + LR"].values, width, label="TF-IDF + LR")
        ax.bar(x + width / 2, pivot["DistilRoBERTa"].values, width, label="DistilRoBERTa")

        ax.set_title(ax_name.replace("_", " "))
        ax.set_ylabel("Micro-F1")
        ax.set_ylim(0.70, 1.00)

        ax.set_xticks(x)
        ax.set_xticklabels([str(v) for v in values])

        ax.tick_params(axis="x", pad=8)

        for j, v in enumerate(values):
            ax.text(
                j, -0.16, f"n={int(n_base.iloc[j])}",
                transform=ax.get_xaxis_transform(),
                ha="center", va="top", fontsize=9,
                clip_on=False
            )

    for j in range(len(axes_present), len(axs)):
        axs[j].axis("off")

    handles, labels = axs[0].get_legend_handles_labels()
    fig.legend(
        handles, labels,
        loc="upper center",
        ncol=2,
        frameon=False,
        bbox_to_anchor=(0.5, 1.03)
    )

    # reserve space for legend + for the below-axis n labels
    fig.subplots_adjust(top=0.88, bottom=0.10, wspace=0.25, hspace=0.35)

    if out_path is not None:
        fig.savefig(out_path, dpi=300, bbox_inches="tight")
        print("Saved:", out_path)

    plt.show()


plot_axis_microf1_comparison_all(
    axis_table,
    axis_table_transformer,
    out_path="figures/microf1_by_axis_baseline_vs_distilroberta.png"
)


In [ ]:
# threshold sweep plot 

def microf1_threshold_sweep(probs: np.ndarray, Y_true: np.ndarray, thresholds):
    probs = np.asarray(probs)
    Y_true = np.asarray(Y_true).astype(int)
    f1s = []
    for t in thresholds:
        Y_pred = (probs >= t).astype(int)
        f1s.append(f1_score(Y_true, Y_pred, average="micro", zero_division=0))
    return np.asarray(f1s)

# 1) probabilities from logits
probs_tr = 1 / (1 + np.exp(-logits))   # sigmoid, shape (N,4)

# 2) sweep
thresholds = np.linspace(0.05, 0.95, 19)
f1_tr = microf1_threshold_sweep(probs_tr, Y_test_true, thresholds)

best_t = thresholds[np.argmax(f1_tr)]
best_f1 = f1_tr.max()
print(f"Transformer best micro-F1 = {best_f1:.4f} at t = {best_t:.2f}")

# 3) save
out_dir = Path("results/threshold_sweeps")
out_dir.mkdir(parents=True, exist_ok=True)

np.savez(
    out_dir / "threshold_sweep_distilroberta.npz",
    thresholds=thresholds,
    f1=f1_tr,
    probs=probs_tr,      
    Y_true=Y_test_true       
)

print("Saved:", out_dir / "threshold_sweep_distilroberta.npz")


In [ ]:
# plotting the 2 models against each other

base_path = Path("results/threshold_sweeps/threshold_sweep_baseline.npz")
tr_path   = Path("results/threshold_sweeps/threshold_sweep_distilroberta.npz")

base = np.load(base_path, allow_pickle=True)
tr   = np.load(tr_path,   allow_pickle=True)

t_base, f1_base = base["thresholds"], base["f1"]
t_tr,   f1_tr   = tr["thresholds"],   tr["f1"]

# sanity check: same threshold grid
if not np.allclose(t_base, t_tr):
    raise ValueError("Threshold grids differ between saved sweeps.")

t = t_base

plt.figure(figsize=(7.2, 4.6))
plt.plot(t, f1_base, marker="o", label="TF-IDF + LR")
plt.plot(t, f1_tr,   marker="o", label="DistilRoBERTa")

plt.axvline(0.50, linestyle="--", linewidth=1)

plt.xlabel("Decision threshold")
plt.ylabel("Micro-F1")
plt.ylim(0.70, 1.00)
plt.title("Threshold sweep on test set (micro-F1)")
plt.legend()
plt.tight_layout()
plt.show()


# save
fig_dir = Path("figures")
fig_dir.mkdir(parents=True, exist_ok=True)
out_fig = fig_dir / "threshold_sweep_microf1.png"
plt.savefig(out_fig, dpi=300, bbox_inches="tight")
print("Saved:", out_fig)
